In [16]:
from enum import Enum
from typing import Any, Literal, Optional, List
from pydantic import BaseModel, Field, PositiveInt, model_validator
from typing_extensions import Self
from openai import OpenAI

In [17]:
client = OpenAI()

In [2]:
class FilterOperator(str, Enum):
    # TODO: Support =, !=, >, <, >=, <=
    EQ = "="
    NE = "!="
    GT = ">"
    LT = "<"
    GE = ">="
    LE = "<="

In [3]:
class Filter(BaseModel):
    # TODO: Define field, operator and value
    field: str = Field(..., description="The name of the field to filter on")
    operator: FilterOperator = Field(..., description="The comparison operator")
    value: str | int | float = Field(..., description="The value to compare against")

In [4]:
class RequestStatus(str, Enum):
    SUPPORTED = "SUPPORTED"
    UNSUPPORTED = "UNSUPPORTED"
    AMBIGUOUS = "AMBIGUOUS"

In [5]:
class Aggregation(str, Enum):
    SUM = "SUM"
    AVG = "AVG"
    COUNT = "COUNT"
    MIN = "MIN"
    MAX = "MAX"

In [6]:
class SortDirection(str, Enum):
    ASC = "ASC"
    DESC = "DESC"

In [7]:
class Sort(BaseModel):
    field: str = Field(..., description="The field name to sort results by")
    direction: SortDirection = Field(..., description="The direction of sorting")

In [8]:
class AnalyticalIntent(BaseModel):
    status: RequestStatus = Field(...)
    metric: Optional[str] = Field(None)
    aggregation: Optional[Aggregation] = Field(None)
    dimensions: List[str] = Field(default_factory=list)
    filters: List[Filter] = Field(default_factory=list)
    sort: Optional[Sort] = Field(None)
    limit: Optional[PositiveInt] = Field(None)
    reason: Optional[str] = Field(None)

    @model_validator(mode="after")
    def validate_intent(self) -> Self:
        # TODO 1: Check required fields for SUPPORTED status
        if self.status == RequestStatus.SUPPORTED:
            if self.metric is None:
                raise ValueError("`metric` must not be None when status is SUPPORTED")
            if self.aggregation is None:
                raise ValueError("`aggregation` must not be None when status is SUPPORTED")

        # TODO 2: Check required reasoning for AMBIGUOUS or UNSUPPORTED status
        if self.status in (RequestStatus.AMBIGUOUS, RequestStatus.UNSUPPORTED):
            if not self.reason or not self.reason.strip():
                raise ValueError(f"`reason` must be a non-empty string when status is {self.status.value}")

        # TODO 3: Return self when validation succeeds
        return self

In [9]:
# Test 1: Valid numeric filter
f1 = Filter(field="amount", operator=">", value=1000)
print(f1)

field='amount' operator=<FilterOperator.GT: '>'> value=1000


In [10]:
# Test 2: Valid string filter
f2 = Filter(field="status", operator="=", value="completed")
print(f2)

field='status' operator=<FilterOperator.EQ: '='> value='completed'


In [11]:
try:
    f3 = Filter(field="amount", operator="LIKE", value=1000)
    print(f3)
except Exception as e:
    print("Validation failed:", e)

Validation failed: 1 validation error for Filter
operator
  Input should be '=', '!=', '>', '<', '>=' or '<=' [type=enum, input_value='LIKE', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


In [12]:
intent = AnalyticalIntent(
    status="SUPPORTED",
    metric="revenue",
    aggregation="SUM",
    dimensions=["customer_id"],
    filters=[
        {"field": "status", "operator": "=", "value": "completed"}
    ],
    sort={"field": "revenue", "direction": "DESC"},
    limit=5
)

print(intent.model_dump_json(indent=2))

{
  "status": "SUPPORTED",
  "metric": "revenue",
  "aggregation": "SUM",
  "dimensions": [
    "customer_id"
  ],
  "filters": [
    {
      "field": "status",
      "operator": "=",
      "value": "completed"
    }
  ],
  "sort": {
    "field": "revenue",
    "direction": "DESC"
  },
  "limit": 5,
  "reason": null
}


In [13]:
ambiguous = AnalyticalIntent(
    status="AMBIGUOUS",
    reason="The user did not specify which metric to analyze."
)

print(ambiguous.model_dump_json(indent=2))

{
  "status": "AMBIGUOUS",
  "metric": null,
  "aggregation": null,
  "dimensions": [],
  "filters": [],
  "sort": null,
  "limit": null,
  "reason": "The user did not specify which metric to analyze."
}


In [14]:
from pydantic import ValidationError

try:
    AnalyticalIntent(
        status="SUPPORTED",
        metric="revenue",
        aggregation="SUM",
        limit=-5
    )
except ValidationError as e:
    print("Correctly rejected:", e)

Correctly rejected: 1 validation error for AnalyticalIntent
limit
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [15]:

from pydantic import ValidationError

test_cases = [
    {
        "status": "SUPPORTED",
        "metric": "revenue",
        "aggregation": "SUM"
    },
    {
        "status": "SUPPORTED",
        "metric": None,
        "aggregation": "SUM"
    },
    {
        "status": "SUPPORTED",
        "metric": "revenue",
        "aggregation": None
    },
    {
        "status": "AMBIGUOUS",
        "reason": "   "
    },
    {
        "status": "UNSUPPORTED",
        "reason": "Non-aggregated queries are outside the current scope."
    }
]

for i, data in enumerate(test_cases, start=1):
    try:
        result = AnalyticalIntent.model_validate(data)
        print(f"Test {i}: ACCEPTED")
    except ValidationError as e:
        print(f"Test {i}: REJECTED - {e.errors()[0]['msg']}")



Test 1: ACCEPTED
Test 2: REJECTED - Value error, `metric` must not be None when status is SUPPORTED
Test 3: REJECTED - Value error, `aggregation` must not be None when status is SUPPORTED
Test 4: REJECTED - Value error, `reason` must be a non-empty string when status is AMBIGUOUS
Test 5: ACCEPTED


In [18]:
SYSTEM_PROMPT = """
You are an analytical intent planner.

Convert the user's analytical question into a structured plan.

Rules:
1. Only plan aggregation-based analytical queries.
2. Extract metric, aggregation, dimensions, filters,
   sorting and limit when applicable.
3. Never generate SQL.
4. If the question is unclear, return AMBIGUOUS
   with a reason.
5. If the request is outside supported analytical
   operations, return UNSUPPORTED with a reason.
6. Do not invent filters or dimensions that the
   user did not request.
"""

In [22]:
def generate_intent(question: str) -> AnalyticalIntent:
    """
    Convert a natural-language question into
    a validated AnalyticalIntent object.
    """
    response = client.responses.parse(
        model="gpt-6-luna",  # Or your chosen model that supports Structured Outputs
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        text_format=AnalyticalIntent
    )

    parsed_intent = response.output_parsed

    if parsed_intent is None:
        raise ValueError("Failed to parse structured analytical intent from response.")

    return parsed_intent

In [23]:
question = "Show the top 5 customers by total revenue"

intent = generate_intent(question)
print(intent.model_dump_json(indent=2))

{
  "status": "SUPPORTED",
  "metric": "revenue",
  "aggregation": "SUM",
  "dimensions": [
    "customer"
  ],
  "filters": [],
  "sort": {
    "field": "revenue",
    "direction": "DESC"
  },
  "limit": 5,
  "reason": null
}


In [24]:

test_questions = [
    {
        "id": "T1",
        "question": "Show the top 5 customers by total revenue",
        "expected_status": "SUPPORTED"
    },
    {
        "id": "T2",
        "question": "Show me the best performing customers",
        "expected_status": "AMBIGUOUS"
    },
    {
        "id": "T3",
        "question": "Show all individual orders placed yesterday",
        "expected_status": "UNSUPPORTED"
    }
]

for test in test_questions:
    intent = generate_intent(test["question"])

    actual_status = intent.status.value

    print(f"\nTest: {test['id']}")
    print(f"Question: {test['question']}")
    print(f"Expected: {test['expected_status']}")
    print(f"Actual: {actual_status}")
    print(f"Pass: {actual_status == test['expected_status']}")
    print(intent.model_dump_json(indent=2))



Test: T1
Question: Show the top 5 customers by total revenue
Expected: SUPPORTED
Actual: SUPPORTED
Pass: True
{
  "status": "SUPPORTED",
  "metric": "revenue",
  "aggregation": "SUM",
  "dimensions": [
    "customer"
  ],
  "filters": [],
  "sort": {
    "field": "revenue",
    "direction": "DESC"
  },
  "limit": 5,
  "reason": null
}

Test: T2
Question: Show me the best performing customers
Expected: AMBIGUOUS
Actual: AMBIGUOUS
Pass: True
{
  "status": "AMBIGUOUS",
  "metric": null,
  "aggregation": null,
  "dimensions": [
    "customer"
  ],
  "filters": [],
  "sort": null,
  "limit": null,
  "reason": "“Best performing” is unclear without a performance metric, such as revenue or number of orders."
}

Test: T3
Question: Show all individual orders placed yesterday
Expected: UNSUPPORTED
Actual: UNSUPPORTED
Pass: True
{
  "status": "UNSUPPORTED",
  "metric": null,
  "aggregation": null,
  "dimensions": [],
  "filters": [],
  "sort": null,
  "limit": null,
  "reason": "This requests ind